# 지형자료는 어떤 자료인가?

영상의 밴드·단위·좌표계와 서버 계산의 의미 확인

**할 일** · 서울 주변 SRTM의 높이 지도와 영역 평균 확인

위에서 아래로 실행. 코드를 바꾸기 전 결과를 예상하고, 실행 결과와 비교

## 실행 준비

CPU 런타임 사용. Earth Engine에 등록한 Cloud 프로젝트와 사용 권한 필요. [설정 안내](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md) 참고. 데이터 조회에는 본인 계정 인증 필요

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Earth Engine에 등록한 본인의 Cloud 프로젝트 ID
if not PROJECT_ID.strip():
    raise ValueError("PROJECT_ID에 본인의 등록된 Cloud 프로젝트 ID 입력")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

## 밴드·해상도와 평균

SRTM은 2000년 관측 기반 지형자료. `elevation`의 단위는 m. 자료의 관측 시점과 오늘의 지표를 구분

In [ ]:
dem = ee.Image('USGS/SRTMGL1_003').select('elevation')
mean_height = dem.reduceRegion(ee.Reducer.mean(), roi, 30, maxPixels=2_000_000)

`mean_height`는 계산을 기술한 서버 객체. 아래 셀의 `getInfo()`에서 작은 결과만 가져옴

In [ ]:
print('bands:', dem.bandNames().getInfo())
print('projection:', dem.projection().getInfo())
print('mean elevation (m):', mean_height.getInfo())

In [ ]:
Map = geemap.Map()
Map.centerObject(roi, 10)
Map.addLayer(dem.clip(roi), {'min': 0, 'max': 800, 'palette': ['white', 'tan', 'darkgreen']}, 'Elevation (m)')
Map.addLayer(roi, {}, 'Study region')
Map

관심 영역을 북쪽 산지로 옮겨 평균 높이를 비교하시오. 영역 평균이 실제 지점의 높이와 다른 이유를 설명하시오.

## 자료와 해석 범위

- [SRTM 30 m](https://developers.google.com/earth-engine/datasets/catalog/USGS_SRTMGL1_003)

관측 연도는 고정 사례의 기간. 수업 연도와 무관. 코드는 자료 처리 방법을 보여 주며 지도·수치는 인증 후 조회 결과로 판단